In [13]:
from langgraph.graph import StateGraph ,START,END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from  dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
import operator

In [14]:
load_dotenv()
generator_llm = ChatGroq(model="openai/gpt-oss-20b")
evaluator_llm = ChatGroq(model="openai/gpt-oss-20b")
optimizer_llm = ChatGroq(model="openai/gpt-oss-20b")

In [15]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal['approved', 'need_improvement'] =Field(...,description='Final evaluation  result')
    feedback: str = Field(..., description="Constructive feedback for the tweet")

In [16]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [17]:
class TweetState(TypedDict):
    
    topic: str
    tweet: str
    evaluation: Literal['approved',"need_improvement"]
    feedback: str
    iteration: int
    max_iteration: int
    
    tweet_history: Annotated[list[str], operator.add]
    feedback_history: Annotated[list[str], operator.add]
    

In [18]:
def generate_tweet(state: TweetState):
    #prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
            Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

            Rules:
            - Do NOT use question-answer format.
            - Max 280 characters.
            - Use observational humor, irony, sarcasm, or cultural references.
            - Think in meme logic, punchlines, or relatable takes.
            - Use simple, day to day english
            """)
    ]
    #send generator_llm
    response = generator_llm.invoke(messages).content
    
    #return response
    return {'tweet': response , 'tweet_history': [response]}

In [19]:
def evaluate_tweet(state: TweetState):
    
    #prompt
    
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
        Evaluate the following tweet:

        Tweet: "{state['tweet']}"

        Use the criteria below to evaluate the tweet:

        1. Originality – Is this fresh, or have you seen it a hundred times before?  
        2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
        3. Punchiness – Is it short, sharp, and scroll-stopping?  
        4. Virality Potential – Would people retweet or share it?  
        5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

        Auto-reject if:
        - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
        - It exceeds 280 characters
        - It reads like a traditional setup-punchline joke
        - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

        """)
]
    #structure output
    response = structured_evaluator_llm.invoke(messages)
    
    return {'evaluation': response.evaluation,'feedback': response.feedback , 'feedback_history': [response.feedback]}

In [20]:
def optimize_tweet(TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
            Improve the tweet based on this feedback:
            "{state['feedback']}"

            Topic: "{state['topic']}"
            Original Tweet:
            {state['tweet']}

            Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
            """)
        ]
    
    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration']+1
    
    return {'tweet': response, 'iteration': iteration} 


In [21]:
def route_evaluation(state: TweetState):

    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [22]:
graph = StateGraph(TweetState)

graph.add_node('generate',generate_tweet)
graph.add_node('evaluate',evaluate_tweet)
graph.add_node('optimize',optimize_tweet)

graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')

graph.add_conditional_edges('evaluate',route_evaluation, {'approved': END,'need_improvement': 'optimize'})
graph.add_edge('optimize','evaluate')

workflow = graph.compile()


In [23]:
initial_state = {
    "topic": "Egdbeq",
    "iteration": 1,
    'max_iteration': 5
}

workflow.invoke(initial_state)

{'topic': 'Egdbeq',
 'tweet': 'Egdbeq is that moment when you type “lol” and your phone autocorrects to “Egdbeq”, then you realize you’ve invented a new emoji for “I’m clueless”. #TechLife #NewWords',
 'evaluation': 'approved',
 'feedback': 'Nice play on autocorrect mishaps—fresh enough for most feeds. Tighten the punch a bit: trim the extra wording or add a snappy twist at the end. Keep the hashtags, but maybe swap one for a trending tech meme tag to boost shareability.',
 'iteration': 1,
 'max_iteration': 5,
 'tweet_history': ['Egdbeq is that moment when you type “lol” and your phone autocorrects to “Egdbeq”, then you realize you’ve invented a new emoji for “I’m clueless”. #TechLife #NewWords'],
 'feedback_history': ['Nice play on autocorrect mishaps—fresh enough for most feeds. Tighten the punch a bit: trim the extra wording or add a snappy twist at the end. Keep the hashtags, but maybe swap one for a trending tech meme tag to boost shareability.']}